In [1]:
import tensorflow as tf
import numpy as np
import os
import time

In [2]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Длина текста: {len(text)}')

Длина текста:1115394


In [6]:
print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [7]:
vocab = sorted(set(text))
print(f'уникальные символы {len(vocab)}')

уникальные символы 65


Векторизация текста

In [8]:
example_texts = ['abcdefg', 'xyz']

chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
chars
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)

In [9]:
ids = ids_from_chars(chars)
ids

<tf.RaggedTensor [[40, 41, 42, 43, 44, 45, 46], [63, 64, 65]]>

In [10]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

In [11]:
chars = chars_from_ids(ids)
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

In [13]:
tf.strings.reduce_join(chars, axis=-1).numpy()
def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

подготовка выборки

In [14]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
for ids in ids_dataset.take(10):
  print(chars_from_ids(ids).numpy().decode('utf-8'))

F
i
r
s
t
 
C
i
t
i


In [15]:
seq_length = 10

In [21]:
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

for seq in sequences.take(1):
  print(chars_from_ids(seq))

tf.Tensor([b'F' b'i' b'r' b's' b't' b' ' b'C' b'i' b't' b'i' b'z'], shape=(11,), dtype=string)


In [22]:
for seq in sequences.take(5):
  print(text_from_ids(seq).numpy())

b'First Citiz'
b'en:\nBefore '
b'we proceed '
b'any further'
b', hear me s'


создаём пары (input, label)

In [25]:
def split_input_target(sequence):
  input_text = sequence[:-1]
  target_text = sequence[1:]
  return input_text, target_text

In [27]:
split_input_target(list("SomeText"))

(['S', 'o', 'm', 'e', 'T', 'e', 'x'], ['o', 'm', 'e', 'T', 'e', 'x', 't'])

In [29]:
dataset = sequences.map(split_input_target)

In [30]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'First Citi'
Target: b'irst Citiz'


Создание тренировочных батчей

In [31]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 10), dtype=tf.int64, name=None), TensorSpec(shape=(64, 10), dtype=tf.int64, name=None))>

Модель

In [32]:
vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = 256
rnn_units = 1024

In [63]:
class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      x, states = self.gru(x, training=training)
    else:
      x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

In [64]:
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

In [65]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 10, 66) # (batch_size, sequence_length, vocab_size)


In [66]:
model.summary()

Model: "my_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (64, 10, 256)          │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ ((64, 10, 1024), (64,  │     3,938,304 │
│                                 │ 1024))                 │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (64, 10, 66)           │        67,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,022,850 (15.35 MB)

 Trainable params: 4,022,850 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [67]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

In [68]:
sampled_indices

array([61, 37, 36, 30, 10, 32, 33, 55, 14,  7])

In [69]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'nds of nob'

Next Char Predictions:
 b'vXWQ3STpA,'


In [70]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

Prediction shape:  (64, 10, 66)  # (batch_size, sequence_length, vocab_size)
Mean loss:         tf.Tensor(4.1899576, shape=(), dtype=float32)


In [71]:
tf.exp(example_batch_mean_loss).numpy()

np.float32(66.02)

In [72]:
model.compile(optimizer='adam', loss=loss)

In [76]:
checkpoint_dir = './training_checkpoints'

checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

In [79]:
EPOCHS = 10

history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 1.1431
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 1.1324
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.1225
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.1152
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 1.1076
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.1021
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 1.0969
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.0936
Epoch 9/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.0890
Epoch 10/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 1.0857


Генерация текста

In [80]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    predicted_logits, states = self.model(inputs=input_ids, states=states, return_state=True)
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    predicted_logits = predicted_logits + self.prediction_mask

    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    predicted_chars = self.chars_from_ids(predicted_ids)

    return predicted_chars, states

In [81]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

In [82]:
start = time.time()
states = None
next_char = tf.constant(['ROMEO:'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)
print('\nRun time:', end - start)

ROMEO:
Paint. Come, Kate. Go to the truth to the pawas that woods, to know
Ohe willingly to boy she will be seen to appeach he were more were be thy nobleman sporture with selfsainest thee gracious wrecks a doiton cockress of my breathy ping her; then in more parties to these and every true?

BAPTISTA:
And I am come on to be patience with me an oppoints:
you twelve her hearts, and one thund their decase the right of other sometime
Bark to be many for how my unfold in noses in the spected to be provide
A gentleman to comforted my father when the stone was nothing with all plague; see of a betters will be friar?

Elraige with maid our sightly to supportu, there lioes, any sun hapted.

PEORUMIO:

First Citizen:
He millone, my grace set my theedst way, the contract unto the bishops, year strong.

ADY:
They call you thank you.

CLEOMENES:
Ockables, and I, that she stumbled,
Thereon in mine I will make your fired for an enteration to the strange rather be undone be a just of it to do, and be

In [85]:
start = time.time()
states = None
next_char = tf.constant(['ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result, '\n\n' + '_'*80)
print('\nRun time:', end - start)

tf.Tensor(
[b"ROMEO:\nSo much upost. Where is your\nname; he's im go thas knowl.\nWedwerlike to have them\nThe friends as the your frowning are worn agoor descestions are in none in good tapsting, and great your arms are not to bed to ad cubjessing tide to walk again.\n\nARIEL:\nI can bed Kate\nWhat thoughts her son\nShe may of talet precul him go with wore yea! hie\nAnd guess so far forfeit or being an oath, were there they ortens, from true with health been would duaf.\n\nPETRUCHIO:\nAy, more, sir, would I bring them they are leceed to death, she may most of war, Tybalts will be unpossestsw on my conspatient fain here are they are net unmong'd in two speak, that cares you do feet a thory of the queen and all the table.\n\nPROSPERO:\nWhere she darch and the\ndispossister tearing of the misser,\nAnd fullore to see that he had the best thee better'd him, it shall be seen'd to speak it.\n\nDUKE VINCENTIO:\nYou pay it to do enter.\n\nKATHARINA:\nAy, and but your sore scoles\nas den-eridag